## **Initial setup**

Install Bambu and required packages:

In [ ]:
!echo "deb http://ppa.launchpad.net/git-core/ppa/ubuntu $(cat /etc/os-release | grep UBUNTU_CODENAME | sed 's/.*=//g') main" >> /etc/apt/sources.list.d/git-core.list
!apt-key adv --keyserver keyserver.ubuntu.com --recv-keys A1715D88E1DF1F24
!apt-get update
!apt-get install -y --no-install-recommends build-essential ca-certificates gcc-multilib g++-multilib git libtinfo5 verilator wget
!wget https://release.bambuhls.eu/appimage/bambu-latest.AppImage
!chmod +x bambu-*.AppImage
!ln -sf $PWD/bambu-*.AppImage /bin/bambu
!ln -sf $PWD/bambu-*.AppImage /bin/panda_shell
!ln -sf $PWD/bambu-*.AppImage /bin/spider
!git clone --depth 1 --filter=blob:none --branch dev/panda --sparse https://github.com/ferrandi/PandA-bambu.git
%cd PandA-bambu
!git sparse-checkout set documentation/bambu101
%cd ..
!mv PandA-bambu/documentation/bambu101/* .

#Introduction to High-Level Synthesis

The **High-Level Synthesis (HLS)** process transforms a high-level language description into a hardware design. The input representation can be any high-level language supported by the HLS tool (e.g., **C/C++, FORTRAN**), and the tool takes care of both optimization and translation to the output RTL-level language (e.g., **Verilog** or **VHDL**).

The flow is similar to that of a standard compiler like **GCC** or **Clang**. After preprocessing and generic optimization of the input code, hardware-oriented transformations are applied. Finally, the backend translates the optimized code into a hardware design by:

- Instantiating functional units,
- Scheduling operations in time,
- Binding each operation to hardware resources.

This process enables software developers to write hardware-accelerated code using familiar programming languages and paradigms.

---

**Bambu** is an open-source High-Level Synthesis (HLS) tool developed by [Politecnico di Milano](https://github.com/ferrandi/PandA-bambu). It allows the generation of **RTL hardware designs** from high-level languages, targeting **FPGA** and **ASIC** platforms.

Bambu is designed to be flexible and research-friendly, making it suitable both for academic study and for developing efficient, real-world accelerators from C/C++ code.

Bambu is distributed as a standalone **AppImage**, making it easy to run in a **Colab notebook** or on any Linux machine without requiring complex installation steps.


To synthesize hardware from C code, we will launch `bambu` through the command-line. You can explore all available options and features by printing its help menu.


In [ ]:
!bambu --help

##Input Application

The HLS tool requires an input description of the application to be translated into hardware. The input must define a **single top-level function** that handles all inputs and outputs. This function will be synthesized into hardware by the tool.

In this example, we use a simple **CRC kernel implementation** as the input application.

You can inspect the content of the source file below:


In [ ]:
!cat /basic_usage/icrc.c

## Miniamal set of options

The HLS tool also requires some specification about the output interface to be generated.

### Top level interface

As a standard compiler needs a main function to know where to start, the HLS tool needs a top level interface to be used as the interface of the generated hardware module.  
Following the CRC example, this can be specified through the `--top-fname=icrc1` command line option. This will let the HLS tool know the hardware design interface has to be the same of function `icrc1` from the input representation.

### Target hardware device

The HLS tool needs to know what is the hardware target for which the RTL description needs to be generated since different assumptions are made based on the target device.  
For this example a Xilinx FPGA board has been selected (`xc7z020-1clg484`) and is required through the `--device-name=xc7z020-1clg484-VVD` command line option.  
Furthermore, the required clock period of the generated accelerator needs to be specified through the `--clock-period` option.  
For a target frequency of 200 MHz, `--clock-period=5` has to be used, as Bambu specifies the clock period in nanoseconds.

### Generated design verification

Bambu HLS offers the possibility to run a verification of the output design with respect to the input representation.  
This can be required by passing a testbench file where the top level function is executed.  
The file is passed to the tool through `--generate-tb=testbench.c`.

Below we print the testbench file used to verify the `icrc1` function:


In [ ]:
!cat /basic_usage/testbench.c

## Generating the Accelerator

Now we are ready to generate the hardware design for the `icrc1` function using the following command:


In [ ]:
%cd basic_usage
!bambu icrc.c --top-fname=icrc1 --device-name=xc7z020-1clg484-VVD --clock-period=5 --generate-tb=testbench.c --simulate -v4
%cd ..

# Output hardware design
After the High-Level Synthesis process completes successfully, the generated RTL design can be found in the `/basic_usage/icrc1.v` file.

Additionally, the HLS tool produces a synthesis script named `/basic_usage/synthesize_Synthesis_icrc1.sh` alongside the Verilog description. This script can be used to run the vendor-specific synthesis tool to generate the bitstream for the target FPGA.


Finally, we can examine the top-level module and inspect its interface.  
We expect to find a `start_port` to trigger the accelerator, along with `done_port`, `clock` and `reset` ports to manage its operation.  
Additionally, there should be a `return_port` for the function result and two separate wires corresponding to the two input parameters.


In [ ]:
import re

def print_last_verilog_interface(filename):
    with open(filename, 'r') as f:
        content = f.read()

    modules = re.findall(r'(module\s+\w+\s*\([^;]*?\);)', content, re.DOTALL)

    if not modules:
        print(f"No modules found in {filename}")
        return

    last_module = modules[-1]

    print(f"Interface of the top module in {filename}:\n")
    print(last_module)

print_last_verilog_interface('basic_usage/icrc1.v')